In [3]:
mgrs_db_path = '/mnt/aurora-r0/jungkyo/OPERA/DSWx-S1-final-patch/shared/input_dir/ancillary_data/MGRS_tile.sqlite'

In [5]:
cases = {
  # "Mining": [
  #   {"lat": -5.773196381012654, "lon": -56.806566069338245, "date": "2025-06-29"},
  #   {"lat": -12.9890141517277,  "lon": -70.06357428871692, "date": "2025-04-18"},
  #   {"lat":  2.0185043672215914, "lon":  29.773492831188207, "date": "2024-12-02"},
  #   {"lat":  0.5632783027227907, "lon": 127.91000858625176, "date": "2024-10-06"},
  # ],
  "Logging": [
    {"lat": -7.614112272029289,  "lon":  -75.40825653555396, "date": "2024-09-06"},
    {"lat":  2.974193399509278,  "lon":   17.879592039849456, "date": "2025-02-28"},
    {"lat": 46.126040790060436,  "lon": -122.42122283481473, "date": "2024-11-30"},
    {"lat": 61.522819402112866,  "lon":   15.973959187506068, "date": "2025-05-15"},
  ],
  "Road expansion": [
    {"lat": 35.42336004982615, "lon": 119.40456071405788, "date": "2025-04-30"},
    {"lat": 34.90684358345317, "lon": 115.46110157851172, "date": "2025-04-13"},
  ],
  "New construction": [
    {"lat": 38.89944199257163, "lon": -76.72312133749506, "date": "2024-12-31"},
  ],
  # "Shifting cultivation": [
  #   {"lat": 16.2567117223014,   "lon": 106.49952368333953},
  #   {"lat": -0.20964738432435112,"lon":  24.247199624105463},
  # ],
  # "High water year": [
  #   {"lat": 11.846950508764863, "lon":  -8.481072476128617},
  # ],
  # "Fire": [
  #   {"lat": 65.55141471615501,  "lon": -125.04737814915957},
  #   {"lat": 40.930464902805944, "lon":  -8.10888146754445},
  # ],
  # "Tornado": [
  #   {"lat": 37.06304145409962,  "lon": -84.31882138592562},
  #   {"lat": 31.381599140402475, "lon": -89.94234828078527},
  #   {"lat": 33.23458722044894,  "lon": -87.99217882878997},
  # ],
  # "Landslide": [
  #   {"lat": -8.21177070438279,  "lon": -75.91228172410337},
  # ],
  # "Dry conditions": [
  #   {"lat": -5.340348096784305, "lon": -37.6704070604747},
  # ],
}

In [7]:
from pathlib import Path
import re
import geopandas as gpd
# import fiona3
from shapely.geometry import Point

def _pick_mgrs_layer(db_path: str) -> str:
    """Pick the most likely layer containing MGRS tiles."""
    layers = fiona.listlayers(db_path)
    if not layers:
        raise RuntimeError("No layers found in the SQLite database.")
    # Prefer names containing 'mgrs' or 'tile'
    for pat in [r'mgrs', r'tile']:
        for lyr in layers:
            if re.search(pat, lyr, re.IGNORECASE):
                return lyr
    return layers[0]  # fallback

def _pick_id_column(cols) -> str | None:
    """Guess a good column for the tile/granule ID."""
    candidates = ['mgrs_tile', 'tile', 'tile_id', 'tileid', 'name', 'grid_id', 'grid', 'id']
    cols_lower = [c.lower() for c in cols]
    for cand in candidates:
        if cand in cols_lower:
            return cols[cols_lower.index(cand)]
    return None

def find_mgrs_tile(lat: float,
                   lon: float,
                   db_path: str,
                   layer: str | None = None) -> dict:
    """
    Return the row dict for the MGRS tile polygon that COVERS the input lat/lon.
    If multiple match (rare), returns the first. Raises if none found.
    """
    db_path = str(Path(db_path))
    lyr = layer #or _pick_mgrs_layer(db_path)

    # Read polygons
    gdf = gpd.read_file(db_path, layer=lyr, driver="SQLite")
    if gdf.empty:
        raise RuntimeError(f"Layer '{lyr}' is empty.")

    if gdf.crs is None:
        # Most MGRS tile DBs are in EPSG:4326; if yours isn’t, set it explicitly.
        # You can replace with the correct CRS if known (e.g., 'EPSG:4326').
        gdf.set_crs("EPSG:4326", inplace=True)

    # Build the point and project to layer CRS
    pt = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(gdf.crs).iloc[0]

    # Fast bbox pre-filter with sindex
    if gdf.sindex:
        cand_idx = list(gdf.sindex.query(pt, predicate="intersects"))
        cand = gdf.iloc[cand_idx]
    else:
        cand = gdf

    # Use 'covers' to include boundary points (safer than 'contains' for tile edges)
    try:
        hit = cand[cand.covers(pt)]
    except Exception:
        # Older geopandas might not have .covers predicate; fall back to intersects
        hit = cand[cand.intersects(pt)]

    if hit.empty:
        raise ValueError(f"No MGRS tile found covering lat={lat}, lon={lon} in layer '{lyr}'.")

    row = hit.iloc[0]
    row_dict = row.drop(labels="geometry").to_dict()
    row_dict["_layer"] = lyr
    row_dict["_crs"] = str(gdf.crs)
    row_dict["_geometry"] = row.geometry  # keep geometry if you need it later
    # Add a normalized 'tile_id' convenience key if we can guess it
    id_col = _pick_id_column(hit.columns)
    if id_col:
        row_dict["tile_id"] = row[id_col]
    return row_dict

# --- example usage ---
mgrs_db_path = "/mnt/aurora-r0/jungkyo/OPERA/DSWx-S1-final-patch/shared/input_dir/ancillary_data/MGRS_tile.sqlite"
result = find_mgrs_tile(lat=34.05, lon=-118.25, db_path=mgrs_db_path, layer='mgrs_tile')
print("Tile:", result.get("tile_id", "<unknown>"))


/mnt/aurora-r0/jungkyo/tool/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver SQLite does not support open option DRIVER
  return ogr_read(


Tile: 11SLT


In [ ]:
from pathlib import Path
import pandas as pd

# Use the helper I gave earlier (make sure it's in scope / imported)
# from your_module import find_mgrs_tile

mgrs_db_path = "/mnt/aurora-r0/jungkyo/OPERA/DSWx-S1-final-patch/shared/input_dir/ancillary_data/MGRS_tile.sqlite"

def _norm_lon(lon):
    # keep in [-180, 180)
    return ((lon + 180.0) % 360.0) - 180.0

rows = []
for category, pts in cases.items():  # <-- uses your existing `cases` dict
    for i, pt in enumerate(pts, 1):
        lat = float(pt["lat"])
        lon = _norm_lon(float(pt["lon"]))
        try:
            hit = find_mgrs_tile(lat=lat, lon=lon, db_path=mgrs_db_path)
            tile_id = hit.get("tile_id")
            # If the helper couldn't guess an ID column, try a fallback:
            if tile_id is None:
                # prefer any field containing 'tile' or 'mgrs'
                k = next((k for k in hit.keys() if isinstance(k, str) and
                          (("tile" in k.lower()) or ("mgrs" in k.lower())) and k not in {"_layer","_crs","_geometry"}), None)
                tile_id = hit.get(k) if k else None

            rows.append({
                "category": category,
                "index_in_category": i,
                "lat": lat,
                "lon": lon,
                "tile_id": tile_id,
                "layer": hit.get("_layer"),
                "crs": hit.get("_crs"),
            })
        except Exception as e:
            rows.append({
                "category": category,
                "index_in_category": i,
                "lat": lat,
                "lon": lon,
                "tile_id": None,
                "layer": None,
                "crs": None,
                "error": str(e),
            })

df = pd.DataFrame(rows)
df


,category,index_in_category,lat,lon,tile_id,layer,crs
0,Mining,1,-5.773196,-56.806566,21MWP,mgrs_tile,EPSG:4326
1,Mining,2,-12.989014,-70.063574,19LCF,mgrs_tile,EPSG:4326
2,Mining,3,2.018504,29.773493,35NRC,mgrs_tile,EPSG:4326
3,Mining,4,0.563278,127.910009,52NCF,mgrs_tile,EPSG:4326
4,Logging,1,-7.614112,-75.408257,18MVS,mgrs_tile,EPSG:4326
5,Logging,2,2.974193,17.879592,33NZD,mgrs_tile,EPSG:4326
6,Logging,3,46.126041,-122.421223,10TES,mgrs_tile,EPSG:4326
7,Logging,4,61.522819,15.973959,33VWJ,mgrs_tile,EPSG:4326
8,Road expansion,1,35.423360,119.404561,50SQE,mgrs_tile,EPSG:4326
9,Road expansion,2,34.906844,115.461102,50SLD,mgrs_tile,EPSG:4326


In [ ]:
# import subprocess
# import os
# import shlex
# from datetime import datetime
# # Common parameters for all runs
# base_cmd = [
#     "dist-s1", "run",
#     "--mgrs_tile_id", "19HBD",
#     "--post_date", "2024-02-16",
#     "--track_number", "18",
#     "--memory_strategy", "high",
#     "--device", "cpu",
#     "--n_workers_for_norm_param_estimation", "4",
#     "--apply_water_mask", "true",
#     "--input_data_dir", "out2"
# ]

# # Different configurations for each run
# configs = [
#     {
#         "dst_dir": "chile_transformer_optimized",
#         "max_pre_imgs": [4, 3, 3],
#         "model_source": "transformer_optimized"
#     },
#     {
#         "dst_dir": "chile_transformer_optimized_fine",
#         "max_pre_imgs": [4, 3, 3],
#         "model_source": "transformer_optimized_fine"
#     },
#     {
#         "dst_dir": "chile_transformer_anni_20",
#         "max_pre_imgs": [8, 6, 6],
#         "model_source": "transformer_anniversary_trained"
#     },
#     {
#         "dst_dir": "chile_transformer_anni_10",
#         "max_pre_imgs": [4, 3, 3],
#         "model_source": "transformer_anniversary_trained_10"
#     },
#     {
#         "dst_dir": "chile_transformer_anni_20_diego_4",
#         "max_pre_imgs": [8, 6, 6],
#         "model_source": "external",
#         "model_cfg_path": "/mnt/aurora-r0/jungkyo/tool/DIST-S1/dist-s1-model/dist-s1-model/model_data/config_4x4_v1_sas.yaml",
#         "model_wts_path": "/mnt/aurora-r0/jungkyo/tool/DIST-S1/dist-s1-model/dist-s1-model/model_data/checkpoint_4x4_v1.pth"
#     },
#     {
#         "dst_dir": "chile_transformer_anni_20_diego_8",
#         "max_pre_imgs": [8, 6, 6],
#         "model_source": "external",
#         "model_cfg_path": "/mnt/aurora-r0/jungkyo/tool/DIST-S1/dist-s1-model/dist-s1-model/model_data/config_8x8_v1_sas.yaml",
#         "model_wts_path": "/mnt/aurora-r0/jungkyo/tool/DIST-S1/dist-s1-model/dist-s1-model/model_data/checkpoint_8x8_v1.pth"

#     }
# ]
# os.makedirs("run_logs", exist_ok=True)

# # Loop through each configuration and run the command
# failures = []
# for cfg in configs:
#     # Ensure output dir exists (harmless if dist-s1 creates it anyway)
#     os.makedirs(cfg["dst_dir"], exist_ok=True)

#     cmd = base_cmd + [
#         "--dst_dir", cfg["dst_dir"],
#         "--max_pre_imgs_per_burst_mw", ",".join(map(str, cfg["max_pre_imgs"])),
#         "--model_source", cfg["model_source"],
#     ]

#     # Add external model args when requested
#     if cfg["model_source"] == "external":
#         for k in ("model_cfg_path", "model_wts_path"):
#             if k not in cfg:
#                 raise ValueError(f"Missing '{k}' for external model in {cfg['dst_dir']}")
#         cmd += ["--model_cfg_path", cfg["model_cfg_path"],
#                 "--model_wts_path", cfg["model_wts_path"]]

#     stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#     log_path = os.path.join("run_logs", f"{os.path.basename(cfg['dst_dir'])}_{stamp}.log")

#     print("\n=== Running ===")
#     print(" ".join(shlex.quote(c) for c in cmd))
#     print(f"Logging to: {log_path}")

#     # Stream stdout/stderr to both notebook and a log file
#     with open(log_path, "w") as lf:
#         proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
#         for line in proc.stdout:
#             print(line, end="")
#             lf.write(line)
#         ret = proc.wait()

#     if ret != 0:
#         print(f"❌ Run failed for {cfg['dst_dir']} (exit {ret})")
#         failures.append((cfg["dst_dir"], ret))
#     else:
#         print(f"✅ Completed: {cfg['dst_dir']}")

# if failures:
#     print("\nSome runs failed:")
#     for d, r in failures:
#         print(f"- {d}: exit {r}")
# else:
#     print("\nAll runs completed successfully.")

In [ ]:
# --- SEARCH TRACK + DATE FROM ASF, THEN RUN DIST-S1 ---

import os, shlex, subprocess
from pathlib import Path
from datetime import datetime, timedelta, timezone

import asf_search as asf

# Your helper from earlier must be available:
# from your_module import find_mgrs_tile

mgrs_db_path = "/mnt/aurora-r0/jungkyo/OPERA/DSWx-S1-final-patch/shared/input_dir/ancillary_data/MGRS_tile.sqlite"

def _norm_lon(lon):  # keep in [-180, 180)
    return ((float(lon) + 180.0) % 360.0) - 180.0

def _guess_tile_id(hit: dict) -> str:
    if hit.get("tile_id"):
        return str(hit["tile_id"])
    for k in hit.keys():
        if k in {"_layer","_crs","_geometry"}: 
            continue
        if isinstance(k, str) and any(tok in k.lower() for tok in ("tile","mgrs")) and hit.get(k) is not None:
            return str(hit[k])
    raise KeyError("Could not infer tile id column from DB row.")

def _to_iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S")

def _parse_start_time(p):
    # asf_search returns metadata under p.properties (dict)
    # prefer 'startTime', else use umm start
    props = p.properties or {}
    val = props.get("startTime") or props.get("starttime") or props.get("start") or props.get("sceneDate")
    if not val:
        # Fallback to UMM TemporalCoverage
        val = (p.umm.get("TemporalExtent", {}) or {}).get("RangeDateTime", {}).get("BeginningDateTime")
    return datetime.fromisoformat(val.replace("Z","+00:00")).astimezone(timezone.utc)

def _pick_nearest(products, target_dt):
    if not products:
        return None
    target_dt = target_dt.astimezone(timezone.utc)
    best = min(products, key=lambda pr: abs(_parse_start_time(pr) - target_dt))
    return best

def find_track_and_date(lat: float, lon: float, target_date: str, days_window: int = 20):
    """
    Return (relativeOrbit:int, chosen_date:'YYYY-MM-DD', product_id, dataset_used)
    Preference: OPERA-S1 RTC; fallback to Sentinel-1 IW SLC/GRD if needed.
    """
    lon = _norm_lon(lon)
    wkt_point = f"POINT({lon} {lat})"

    # Build temporal window
    dt0 = datetime.fromisoformat(target_date).replace(tzinfo=timezone.utc)
    start = _to_iso(dt0 - timedelta(days=days_window))
    end   = _to_iso(dt0 + timedelta(days=days_window))
    temporal = f"{start},{end}"

    # 1) Try OPERA RTC-S1 (burst-based) near the point
    # try:
    # res = asf.search(
    #     dataset=[asf.DATASET.OPERA_S1],
    #     processingLevel=[asf.PROCESSING_LEVEL.RTC],
    #     beamMode=[asf.BEAMMODE.IW],
    #     intersectsWith=wkt_point,
    #     temporal=temporal,
    #     maxResults=500,
    # )
    res = asf.geo_search(
        processingLevel='RTC',
        intersectsWith=wkt_point,
        start=start,
        end=end,
        maxResults=500,
    )
    products = list(res)
    # except Exception:
    #     products = []

    # 2) If nothing, fall back to Sentinel-1 (SLC first, then GRD)
    if not products:
        try:
            res = asf.search(
                dataset=[asf.DATASET.SENTINEL1],
                productType=[asf.PRODUCT_TYPE.SLC],
                beamMode=[asf.BEAMMODE.IW],
                intersectsWith=wkt_point,
                temporal=temporal,
                maxResults=500,
            )
            products = list(res)
            fallback_ds = "SENTINEL-1 SLC"
        except Exception:
            products = []

        # if not products:
        #     res = asf.search(
        #         dataset=[asf.DATASET.SENTINEL1],
        #         productType=[asf.PRODUCT_TYPE.GRD],
        #         beamMode=[asf.BEAMMODE.IW],
        #         intersectsWith=wkt_point,
        #         temporal=temporal,
        #         maxResults=500,
        #     )
        #     products = list(res)
        #     fallback_ds = "SENTINEL-1 GRD"
    else:
        fallback_ds = "OPERA-S1 RTC"

    if not products:
        raise RuntimeError("No ASF products found within ±{} days.".format(days_window))

    best = _pick_nearest(products, dt0)
    props = best.properties
    rel_orbit = props.get("relativeOrbit")
    if rel_orbit is None:
        # Some rare CMR records may lack relativeOrbit; as a last resort, parse from OPERA burst ID T### if present
        ob = props.get("operaBurstID") or props.get("fullBurstID")
        if ob and ob.startswith("T") and ob[1:4].isdigit():
            rel_orbit = int(ob[1:4])
        else:
            raise KeyError("relativeOrbit not present in product properties.")

    chosen_dt = _parse_start_time(best).date().isoformat()
    return int(rel_orbit), chosen_dt, props.get("sceneName") or props.get("granuleName") or props.get("fileID"), fallback_ds

# ----------------- Build run plan from your cases -----------------

runs_root = Path("runs_dist_s1")
logs_root = Path("run_logs")
runs_root.mkdir(parents=True, exist_ok=True)
logs_root.mkdir(parents=True, exist_ok=True)

# Your dist-s1 model configs (unchanged)
configs = [
    {"dst_dir": "chile_transformer_optimized",      "max_pre_imgs": [4,3,3], "model_source": "transformer_optimized"},
    {"dst_dir": "chile_transformer_optimized_fine", "max_pre_imgs": [4,3,3], "model_source": "transformer_optimized_fine"},
    {"dst_dir": "chile_transformer_anni_20",        "max_pre_imgs": [8,6,6], "model_source": "transformer_anniversary_trained"},
    {"dst_dir": "chile_transformer_anni_10",        "max_pre_imgs": [4,3,3], "model_source": "transformer_anniversary_trained_10"},
    {"dst_dir": "chile_transformer_anni_20_diego_4","max_pre_imgs": [8,6,6], "model_source": "external",
     "model_cfg_path": "/mnt/aurora-r0/jungkyo/tool/DIST-S1/dist-s1-model/dist-s1-model/model_data/config_4x4_v1_sas.yaml",
     "model_wts_path": "/mnt/aurora-r0/jungkyo/tool/DIST-S1/dist-s1-model/dist-s1-model/model_data/checkpoint_4x4_v1.pth"},
    {"dst_dir": "chile_transformer_anni_20_diego_8","max_pre_imgs": [8,6,6], "model_source": "external",
     "model_cfg_path": "/mnt/aurora-r0/jungkyo/tool/DIST-S1/dist-s1-model/dist-s1-model/model_data/config_8x8_v1_sas.yaml",
     "model_wts_path": "/mnt/aurora-r0/jungkyo/tool/DIST-S1/dist-s1-model/dist-s1-model/model_data/checkpoint_8x8_v1.pth"},
]

# Base cmd WITHOUT --mgrs_tile_id/--post_date/--track_number
base_cmd_common = [
    "dist-s1","run",
    "--memory_strategy","high",
    "--device","cpu",
    "--n_workers_for_norm_param_estimation","4",
    "--apply_water_mask","true",
    "--input_data_dir","out2",
]

failures = []

for category, pts in cases.items():
    for idx, pt in enumerate(pts, 1):
        print(category, pt)

        lat = float(pt["lat"])
        lon = _norm_lon(pt["lon"])
        req_date = str(pt["date"])

        # 1) tile lookup
        try:
            hit = find_mgrs_tile(lat=lat, lon=lon, db_path=mgrs_db_path)
            tile_id = _guess_tile_id(hit)
        except Exception as e:
            print(f"❌ [{category} #{idx}] tile lookup failed: {e}")
            failures.append((f"{category}#{idx}", f"tile_lookup:{e}"))
            continue

        # 2) ASF search for track/date near requested date
        try:
            track, post_date, prod_id, dataset_used = find_track_and_date(lat, lon, req_date, days_window=6)
        except Exception as e:
            print(f"❌ [{category} #{idx}] ASF search failed: {e}")
            failures.append((f"{category}#{idx}", f"asf_search:{e}"))
            continue

        case_tag = f"{category.replace(' ','_')}_{idx}_{tile_id}_T{track}_{post_date}"
        case_root = runs_root / case_tag
        case_root.mkdir(parents=True, exist_ok=True)

        print(f"\n=== Plan for {case_tag} ===")
        print(f"  lat/lon: {lat:.6f}, {lon:.6f}")
        print(f"  tile:    {tile_id}")
        print(f"  track:   {track}  (from {dataset_used}, product={prod_id})")
        print(f"  date:    {post_date} (nearest within ±6d of {req_date})")

        # 3) Run all configs with resolved tile/date/track
        for cfg in configs:
            dst_dir_full = case_root / cfg["dst_dir"]
            dst_dir_full.mkdir(parents=True, exist_ok=True)

            cmd = (
                base_cmd_common
                + ["--mgrs_tile_id", tile_id,
                   "--post_date", post_date,
                   "--track_number", str(track)]
                + ["--dst_dir", str(dst_dir_full),
                   "--max_pre_imgs_per_burst_mw", ",".join(map(str, cfg["max_pre_imgs"])),
                   "--model_source", cfg["model_source"]]
            )

            if cfg["model_source"] == "external":
                cmd += ["--model_cfg_path", cfg["model_cfg_path"],
                        "--model_wts_path", cfg["model_wts_path"]]

            stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            log_path = logs_root / f"{dst_dir_full.name}_{case_tag}_{stamp}.log"

            print("\n--- Running ---")
            print(" ".join(shlex.quote(c) for c in cmd))
            print(f"Log: {log_path}")

            with open(log_path, "w") as lf:
                proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                ret = proc.wait()

            if ret != 0:
                print(f"❌ Run failed for {dst_dir_full.name} (exit {ret})")
                failures.append((f"{case_tag}/{dst_dir_full.name}", ret))
            else:
                print(f"✅ Completed: {dst_dir_full.name}")

# Summary
if failures:
    print("\nSome runs failed:")
    for d, r in failures:
        print(f"- {d}: {r}")
else:
    print("\nAll runs completed successfully.")


Logging {'lat': -7.614112272029289, 'lon': -75.40825653555396, 'date': '2024-09-06'}


/mnt/aurora-r0/jungkyo/tool/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver SQLite does not support open option DRIVER
  return ogr_read(
/tmp/ipykernel_36800/2154269200.py:71: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  res = asf.geo_search(
/tmp/ipykernel_36800/2154269200.py:71: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/pytho


=== Plan for Logging_1_18MVS_T171_2024-09-06 ===
  lat/lon: -7.614112, -75.408257
  tile:    18MVS
  track:   171  (from OPERA-S1 RTC, product=OPERA_L2_RTC-S1_T171-366271-IW3_20240906T103845Z_20240906T153554Z_S1A_30_v1.0)
  date:    2024-09-06 (nearest within ±6d of 2024-09-06)

--- Running ---
dist-s1 run --memory_strategy high --device cpu --n_workers_for_norm_param_estimation 4 --apply_water_mask true --input_data_dir out2 --mgrs_tile_id 18MVS --post_date 2024-09-06 --track_number 171 --dst_dir runs_dist_s1/Logging_1_18MVS_T171_2024-09-06/chile_transformer_optimized --max_pre_imgs_per_burst_mw 4,3,3 --model_source transformer_optimized
Log: run_logs/chile_transformer_optimized_Logging_1_18MVS_T171_2024-09-06_20250826_175149.log
/mnt/aurora-r0/jungkyo/tool/DIST-S1/dist-s1-enumerator/src/dist_s1_enumerator/asf.py:108: UserWarning: No results - please check burst id and availability.
  warn('No results - please check burst id and availability.', category=UserWarning)


Reading tile me

/mnt/aurora-r0/jungkyo/tool/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver SQLite does not support open option DRIVER
  return ogr_read(
/tmp/ipykernel_36800/2154269200.py:71: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  res = asf.geo_search(
/tmp/ipykernel_36800/2154269200.py:71: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/pytho


=== Plan for Logging_2_33NZD_T109_2025-03-01 ===
  lat/lon: 2.974193, 17.879592
  tile:    33NZD
  track:   109  (from OPERA-S1 RTC, product=OPERA_L2_RTC-S1_T109-233036-IW2_20250301T043339Z_20250301T204517Z_S1A_30_v1.0)
  date:    2025-03-01 (nearest within ±6d of 2025-02-28)

--- Running ---
dist-s1 run --memory_strategy high --device cpu --n_workers_for_norm_param_estimation 4 --apply_water_mask true --input_data_dir out2 --mgrs_tile_id 33NZD --post_date 2025-03-01 --track_number 109 --dst_dir runs_dist_s1/Logging_2_33NZD_T109_2025-03-01/chile_transformer_optimized --max_pre_imgs_per_burst_mw 4,3,3 --model_source transformer_optimized
Log: run_logs/chile_transformer_optimized_Logging_2_33NZD_T109_2025-03-01_20250826_185106.log


Reading tile metadata: 100%|████████████████████| 1/1 [00:00<00:00, 136.97it/s]

Reading tile imagery: 100%|██████████████████████| 1/1 [00:00<00:00,  1.84it/s]

Despeckling and serializing RTC S1 files:   0%|        | 0/330 [00:00<?, ?it/s]
Despeckling and 